# 07 — Mine unknown PPI negative candidates with a frozen ProtT5 predictor

**Status:** drafting code for review; no real unknown-pair scores have been produced here.

Run after **00** audits the shared labeled CSVs. This workflow audits the exported
SNOOPPI **unknown** split, scores valid rows with an existing frozen predictor,
and optionally caches a candidate review queue under `mined/`. It never trains,
changes labeled splits, or assigns label 0 to an unknown pair. Every action switch
starts disabled. An observed count replaces the approximate 835k estimate.

SNOOPPI source: [ChatterjeeLab/SNOOPPI](https://huggingface.co/datasets/ChatterjeeLab/SNOOPPI).
Cite Vincoff and Chatterjee, *Bioinformatics* (2026),
[doi:10.1093/bioinformatics/btag736](https://doi.org/10.1093/bioinformatics/btag736).
Retain the dataset revision used for your CSV export; this notebook does not download
a changing dataset revision automatically.

Working labels retain the documented SNOOPPI positive/negative definitions.
Unknown pairs belong to neither class. Model-screened candidates have no confirmed
negative evidence and require review before any future semi-supervised experiment.


## 1. Shared Colab setup

Mount Drive yourself and fill in the same three split paths and PLM cache paths
used by **00–06**. This setup cell is identical across the draft suite.


In [ ]:
from pathlib import Path
import os
import sys

# Code location and data/artifact location are separate. Mount Drive first in Colab:
# from google.colab import drive
# drive.mount("/content/drive")
# Set these environment variables BEFORE running this cell, consistently in every notebook:
# os.environ["SNOOPPI_ROOT"] = "/content/drive/MyDrive/SNOOPPI"
# os.environ["SNOOPPI_SPLIT_DIR"] = "splits"  # or "splits_ablations/cold_cluster_seed44_v2"
# os.environ["SNOOPPI_REPO_ROOT"] = "/content/SNOOPPI"
repository_root = Path(os.environ.get("SNOOPPI_REPO_ROOT", "/content/SNOOPPI"))
candidates = [Path.cwd(), Path.cwd() / "drafting_code", Path.cwd().parent / "drafting_code",
              repository_root / "drafting_code"]
DRAFT_DIR = next((p for p in candidates if (p / "snooppi_utils.py").is_file()), None)
if DRAFT_DIR is None:
    raise FileNotFoundError("Cannot locate drafting_code/snooppi_utils.py; set SNOOPPI_REPO_ROOT to the clone root")
if str(DRAFT_DIR) not in sys.path:
    sys.path.insert(0, str(DRAFT_DIR))

from snooppi_utils import (
    LABEL, SEQUENCE_A, SEQUENCE_B, canonical_pair_id, full_pair_id,
    experiment_paths, load_splits, normalize_sequence, sequence_id,
    full_sequence_id, split_manifest, cache_paths, add_pair_ids,
    assert_model_ready, validate_cache_coverage, preprocessing_report,
    require_comparable_preprocessing, file_sha256, make_run_manifest,
    prediction_identity, validate_prediction_identity, align_paired_predictions,
    prepare_outputs, save_json,
)

PATHS = experiment_paths()

# ===== EDIT YOUR GOOGLE DRIVE FILE PATHS HERE =====
# The uploaded train/validation/test CSVs contain sequences and labels.
# Replace any value below with Path("/content/drive/MyDrive/YOUR_FOLDER/your_file.csv").
# Copy the SAME three CSV paths into every draft notebook. Existing CSVs are read directly;
# this setup does not generate splits or change their contents.
SPLIT_CSV_FILES = {
    "train": PATHS["splits"] / "train.csv",            # TODO: your cached training CSV
    "validation": PATHS["splits"] / "validation.csv",  # TODO: your cached validation CSV
    "test": PATHS["splits"] / "test.csv",              # TODO: your cached test CSV
}

# Existing frozen PLM embeddings are loaded separately from the earlier .pt exports.
# TODO: fill in your own .pt cache paths if the filenames/folders below differ.
# These CSVs are not embedding caches; changing a filename does not convert a file format.
PLM_CACHE_FILES = {
    "ProtT5": PATHS["embeddings"] / "prot_t5_embedding_cache.pt",
    "ESM-2 650M": PATHS["embeddings"] / "esm2_t33_650M_snooppi_mean_pool_embeddings.pt",
    "ESM-C 600M": PATHS["embeddings"] / "esmc_600m_snooppi_mean_pool_embeddings.pt",
}
PATHS["split_files"] = SPLIT_CSV_FILES
PATHS["cache_files"] = PLM_CACHE_FILES
print("Data root:", PATHS["root"])
print("Split CSV files:", SPLIT_CSV_FILES)
print("PLM cache files:", PLM_CACHE_FILES)
# No directories are created until an explicit write switch is enabled.


## 2. Mining paths and explicit stage switches

**Checkpoint correction:** draft **03** runs pair-feature ablations and does not
export the original ProtT5 MLP. The known compatible checkpoint is
`best_predictor_checkpoint.pt` exported by `notebooks/01_prott5_predictor.ipynb`.
If your separate notebook 03 saved that exact architecture, fill in its actual
path below. Other heads fail validation instead of being guessed. The original bare
`best_predictor_weights.pt` export is supported too; use either exact format.
Checkpoint and cache loading share the same trusted-file compatibility rule.

Use only trusted checkpoint/cache files. A legacy checkpoint has no training split
fingerprint, and legacy caches have no encoder revision signature. Review which
split CSVs and preprocessing produced them before setting
`CHECKPOINT_PROVENANCE_REVIEWED=True`. A recorded conflicting split fingerprint
always fails, even after manual review. Conflicting encoder, normalization
or pair-feature-order metadata also fails; split fingerprints are checked
both at the top level and inside `split_manifest` when supplied.

Choose a fresh `MINED_DIR` for a rerun: existing artifacts cannot be overwritten.
Optional extra caches must use original raw sequence keys and the same ProtT5
workflow as the predictor. No encoder download or embedding computation occurs.


In [ ]:
from snooppi_mining import (
    audit_unknown, load_mining_caches, load_predictor, input_record,
    score_unknown, select_candidates, write_audit, write_candidates, cache_verdict,
)
from snooppi_utils import leakage_report
import torch

# ===== EDIT YOUR GOOGLE DRIVE MINING FILE PATHS HERE =====
UNKNOWN_CSV = PATHS["root"] / "raw" / "snooppi_unknown.csv"  # TODO: exported UNKNOWN split CSV
UNKNOWN_DATASET_REVISION = "FILL_IN_HUGGING_FACE_COMMIT_OR_EXPORT_REVISION"  # TODO
CHECKPOINT_PATH = PATHS["checkpoints"] / "best_predictor_checkpoint.pt"  # TODO: original checkpoint OR best_predictor_weights.pt
MINED_DIR = PATHS["root"] / "mined"  # use a new folder for a later run
PROTT5_CACHE_PATHS = [PLM_CACHE_FILES["ProtT5"]]
# If the labeled cache lacks unknown proteins, append your EXISTING compatible cache:
# PROTT5_CACHE_PATHS.append(Path("/content/drive/MyDrive/YOUR_FOLDER/prott5_unknown_cache.pt"))
# Do not substitute ESM vectors, normalize raw keys, or silently skip missing keys.

CHECK_CACHE_COVERAGE = False  # audit may run without loading embedding/checkpoint files
CHECKPOINT_PROVENANCE_REVIEWED = False  # TODO: review source files and historical training provenance
WRITE_AUDIT = False
RUN_SCORING = False  # inference only; writes scores + manifest, never selects candidates
RUN_SELECTION = False  # separate step: reads an already saved and verified score CSV
CACHE_VERDICT = False  # separate step: caches the candidate review verdict

THRESHOLD = 0.1  # strict inequality: score < THRESHOLD
K = None  # all eligible unique pairs; optional positive integer cap
SEED = 44  # recorded for future work; selection ranks deterministically, without random sampling
CHUNK_SIZE = 5_000
INFERENCE_BATCH_SIZE = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SUMMARY_PATH = MINED_DIR / "mining_summary.csv"
MISSING_KEYS_PATH = MINED_DIR / "missing_embedding_keys.csv"
SCORES_PATH = MINED_DIR / "unlabeled_scores.csv"
CANDIDATES_PATH = MINED_DIR / "candidate_negatives.csv"
CONFIG_PATH = MINED_DIR / "mining_config.yaml"
VERDICT_PATH = MINED_DIR / "mining_verdict.json"
print("Unknown CSV:", UNKNOWN_CSV)
print("Checkpoint:", CHECKPOINT_PATH)
print("Mining destination:", MINED_DIR)


## 3. Confirm data mining: audit only

Report total rows, both-sequence validity, duplicate full pairs, and overlap with
train/validation/test. Overlap counts are **valid pair rows** in which either
partner occurs in either labeled role. Counts use full sequences, actual ProtT5
inputs, and historical ESM inputs; the union is not a sum of these counts.

Validity preserves the checkpoint's normalization: uppercase, U/Z/O/B → X,
first 1024 characters. Missing sequences, non-residue markup (including annotated
PTMs), whitespace, non-protein molecule types when supplied, and all-X model
inputs are excluded and counted. These cases need separate reviewed parsing;
they are not silently transformed into ordinary proteins. Source labels, when
present, must be unknown; numeric labels are rejected.

Existing labeled-split problems are displayed rather than repaired here. Mining
does not make historical evaluation results credible retroactively. No fitting
occurs during this audit or any later stage of this notebook.


In [ ]:
splits = load_splits(PATHS)
split_record = split_manifest(PATHS)
print("Selected labeled split fingerprint:", split_record["split_set_sha256"])
display(leakage_report(splits))

cache, cache_records = None, None
if CHECK_CACHE_COVERAGE or RUN_SCORING:
    cache, cache_records = load_mining_caches(PROTT5_CACHE_PATHS)

summary, missing_keys = audit_unknown(UNKNOWN_CSV, splits, cache=cache, chunk_size=CHUNK_SIZE)
display(summary)
print("Unknown export revision:", UNKNOWN_DATASET_REVISION)
print("Unknown CSV SHA-256:", file_sha256(UNKNOWN_CSV))
if cache is not None:
    print("Missing raw-sequence embedding keys:", len(missing_keys))
    if len(missing_keys):
        display(missing_keys.head(10))
        print("Supply a separate compatible cache before scoring; no rows will be dropped.")
if WRITE_AUDIT:
    write_audit(summary, missing_keys, SUMMARY_PATH,
                MISSING_KEYS_PATH if cache is not None else None)
    print("Saved audit:", SUMMARY_PATH)
else:
    print("Audit displayed only. Set WRITE_AUDIT=True to save its summary table.")


## 4. Score every valid unknown pair with the existing frozen predictor

Enable `RUN_SCORING` only after the audit and cache coverage review. The adapter
checks the exact original 3072 → 512 → 512 → 256 → 1 MLP, applies frozen
sum/absolute-difference/product pair features, disables dropout/gradients, and
checks that features and logits are finite before applying sigmoid. Raw
sequence cache keys are preserved. Scores remain continuous; `THRESHOLD`
is the separate mining cutoff rather than a checkpoint classification step.

**Every valid row is scored, including rows overlapping labeled proteins.**
Duplicates are retained at this stage. Output has exactly `pair_id`,
`Partner_A_sequence`, `Partner_B_sequence`, `score`; `pair_id` is the unordered
full-sequence identity (distinct from the truncated-input identity). A separate
manifest records all input/output hashes, row counts, preprocessing, source
revision, and checkpoint provenance. Missing vectors abort the stage and leave
no accepted partial output. Features are constructed one inference batch at a
time, avoiding an 835k × 3072 feature matrix.

Scores are **uncalibrated model outputs**, not demonstrated interaction
probabilities. A single checkpoint cannot supply a defensible per-pair confidence
interval. This draft does not invent one or fit an exponential-tilting model.
Validation calibration, a reviewed ensemble, or a specified density-ratio model
can be designed in a later session before making those claims.


In [ ]:
if RUN_SCORING:
    if cache is None:
        cache, cache_records = load_mining_caches(PROTT5_CACHE_PATHS)
    if len(missing_keys):
        raise ValueError(f"Scoring stopped: {len(missing_keys):,} valid raw sequence keys lack embeddings")
    predictor, checkpoint_record = load_predictor(
        CHECKPOINT_PATH, split_record,
        provenance_reviewed=CHECKPOINT_PROVENANCE_REVIEWED, device=DEVICE,
    )
    record = input_record(UNKNOWN_CSV, split_record, checkpoint_record, cache_records)
    record["unknown_dataset_revision"] = UNKNOWN_DATASET_REVISION
    score_manifest = score_unknown(
        UNKNOWN_CSV, predictor, cache, record, SCORES_PATH,
        chunk_size=CHUNK_SIZE, batch_size=INFERENCE_BATCH_SIZE, device=DEVICE,
    )
    print("Valid rows scored:", score_manifest["valid_scored_rows"])
    print("Scores saved:", SCORES_PATH)
    print("No candidate selection has occurred in this scoring step.")
else:
    print("Scoring disabled. No checkpoint loaded and no scores written.")


## 5. Select candidate negatives as a separate review queue

Enable `RUN_SELECTION` to read the saved `unlabeled_scores.csv` and its manifest.
Inputs and score hashes must still match. A candidate requires both:

1. `score < THRESHOLD` (initially 0.1).
2. Neither partner overlaps any labeled train/validation/test protein under
   full-sequence, ProtT5-input, or ESM-input identity, regardless of partner role.

Deduplicate full unordered pairs, sort by score then identity, and optionally
retain the first `K`. All scored copies of the same full pair must agree,
including copies above the cutoff; inconsistent duplicates stop selection.
No random sampling is performed. This exclusion establishes
identity separation; it does **not** establish MMseqs2 family/cluster isolation.
Use your reviewed homology mapping for any later cold-family claim.

The CSV retains source unknown status, absence of negative evidence, and the
selection threshold. It contains **no binary label**. The YAML config records
threshold, K, seed, checkpoint path/hash, split fingerprint, counts and limits.
Its syntax is JSON, which is a valid YAML 1.2 document and preserves null K and
paths without an extra package. Zero candidates is a valid, reported outcome.


In [ ]:
if RUN_SELECTION:
    candidates, mining_config = select_candidates(
        SCORES_PATH, splits, split_record, threshold=THRESHOLD, K=K, seed=SEED,
        provenance_reviewed=CHECKPOINT_PROVENANCE_REVIEWED, chunk_size=CHUNK_SIZE,
    )
    write_candidates(candidates, mining_config, CANDIDATES_PATH, CONFIG_PATH)
    print("Low-score rows:", mining_config["low_score_rows"])
    print("Low-score rows excluded for labeled protein overlap:", mining_config["excluded_low_score_overlap_rows"])
    print("Eligible unique pairs:", mining_config["eligible_unique_pairs"])
    print("Selected candidate pairs:", mining_config["selected_pairs"])
    display(candidates.head(10))
    print("Saved candidate review queue and config:", CANDIDATES_PATH, CONFIG_PATH)
else:
    print("Candidate selection disabled.")


## 6. Cache the mined-data verdict

Enable `CACHE_VERDICT` only after selection outputs exist. The cached verdict
links the reviewed candidate CSV and config to their hashes and source artifacts.
It is `review_queue_ready` or `no_eligible_candidates`; every biological verdict
remains **unknown; negative evidence not established**. Changed inputs or outputs
invalidate reuse. Candidate artifacts stay under `mined/`, separate from all
supervised and ablation split directories.

Stop here. Semi-supervised retraining, pseudo-labeling, reweighting, exponential
tilting, calibration and incorporation into a new benchmark require a later
session and a separately reviewed protocol. No training is implemented here.


In [ ]:
if CACHE_VERDICT:
    verdict = cache_verdict(CANDIDATES_PATH, CONFIG_PATH, VERDICT_PATH, split_record)
    print("Cached status:", verdict["status"])
    print("Candidate count:", verdict["selected_pairs"])
    print("Biological verdict:", verdict["verdict"])
    print("Saved verdict:", VERDICT_PATH)
else:
    print("Verdict caching disabled. No future training has been performed.")
